# BreastDCEDL_ISPY2
## BreastDCEDL_ISPY2 Data Explore and Visualize
## No Download needed to Use this demo !


#### Author: Tomer Fridman
#### Date: 2025-08-9


In [73]:
from sklearn.metrics import classification_report,auc,roc_auc_score
from PIL import Image
import time
from pathlib import Path


import os
import numpy as np
import pandas as pd
from PIL import Image


import warnings
warnings.filterwarnings('ignore', '.*do not.*', )
warnings.warn('Do not show this message')

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from glob import glob
#from skimage import io
from sklearn.utils import shuffle

#from nipype.interfaces.ants import N4BiasFieldCorrection
import sys
import os
import ast
import re
import nibabel as nib
import warnings
warnings.filterwarnings('ignore')

In [74]:
def show_n_images(
    imgs,
    cmap="gray",
    titles=None,
    enlarge=4,
    mtitle=None,
    cut=False,
    axis_off=True,
    fontsize=15,
    cb=False,
):
    """Visualise *n* images side-by-side."""
    _ = plt.figure(figsize=(enlarge * len(imgs), enlarge * 2))
    for i, im in enumerate(imgs, 1):
        ax = plt.subplot(1, len(imgs), i)
        if cb and len(np.unique(im)) > 5:
            im = cont_br(im)
        ax.imshow(im[50:290, 75:450] if cut else im, cmap=cmap, origin="lower")
        if titles is not None:
            ax.set_title(titles[i - 1], fontsize=fontsize)
        if axis_off:
            ax.axis("off")
    if mtitle:
        plt.suptitle(mtitle, fontsize=fontsize + 2)
    plt.tight_layout(); plt.show()


In [75]:
# --------------------------------------------------------------------- #
#                       NIfTI I/O utilities                             #
# --------------------------------------------------------------------- #
def read_nifti(path: str) -> np.ndarray:
    return nib.load(path).get_fdata()

def _last_int(path):
    m = re.search(r"(\d+)\.nii\.gz$", path)
    return int(m.group(1)) if m else -1


def get_sorted_nifti_acquisitions(pid):

    for part in os.listdir(dce_path):
        path = os.path.join(dce_path, part, pid)
        if os.path.isdir(path):
            files = sorted(f for f in os.listdir(path) if f.endswith('.nii.gz'))
            if deb:
                print("Found:", path)
                print("Files:", files)
            return [read_nifti(os.path.join(path, f)) for f in files]
    return []



def get_nifti_mask(pid: str):

    mpath = mask_path
    if deb: print(mpath)

    ff = [x for x in os.listdir(mpath) if pid in x]

    if deb:print(ff)
    if len(ff)==0: return None

    f=ff[0]

    x = read_nifti(os.path.join(mpath, f))

    return x


In [76]:
def minmax(m):
    if m.max()==0:
        return m
    m = (m-m.min())/(m.max()-m.min())
    return m

def to_rgb(a,b,c):

    stack_axis = a.ndim

    # Stack along the next axis
    im = np.stack([minmax(a), minmax(b), minmax(c)], axis=stack_axis)

    return im

# Setup path's to downloaded data
## If data saved in Google Drive and you run from Colab, mount drive and set path

In [77]:
# set to base of your local nifti data
data_path="."

In [78]:
# Check if running in Google Colab

if 'google.colab' in str(get_ipython()):

    from google.colab import drive
    drive.mount('/content/drive')

    data_path='/content/drive/MyDrive/BreastDCEDL_ISPY2_dataset/'
    

In [79]:
# Set up your local nifti path's

dce_path= os.path.join(data_path,"dce")

mask_path=os.path.join(data_path,"BreastDCEDL_ISPY2_mask")

In [80]:
!pwd

/c/Users/naomi/Downloads/breast_mri/BreastDCEDL_ISPY2_dataset


In [81]:
!ls

BreastDCEDL_ISPY2 NoN pCR dataset .pdf
BreastDCEDL_ISPY2 â€“  Full Dataset.pdf
BreastDCEDL_ISPY2_mask
BreastDCEDL_ISPY2_mask.tar.gz
BreastDCEDL_ISPY2_metadata.csv
BreastDCEDL_ISPY2_nopcr
BreastDCEDL_ISPY2_pcr
BreastDCEDL_dataset.pdf
BrestDCEDL_ISPY2_demo_on_local_data.ipynb
BrestDCEDL_ISPY2_demo_on_local_data_min_crop_with_grid (2).ipynb
BrestDCEDL_ISPY2_zenodo_demo.ipynb
BrestDCEDL_ISPY2_zenodo_demo_grid.ipynb
dce
ispy2_dce_grid_41.png
ispy2_dce_grid_42.png
ispy2_dce_grid_5.png
ispy2_dce_grid_77.png
ispy2_dce_rgb_grid.png
ispy2_dce_rgb_grid_2.png
ispy2_dce_rgb_grid_3.png
ispy2_dce_rgb_grid_4.png
ispy2_dce_rgb_grid_8.png
ispy2_dce_rgb_grid_99.png
ispy2_dce_timepoints.png
ðŸ“� BreastDCEDL_ISPY2 pCR dataset .pdf


## Load and view metada

In [82]:
df=pd.read_csv('BreastDCEDL_ISPY2_metadata.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 982 entries, 0 to 981
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pid             982 non-null    object 
 1   pCR             982 non-null    float64
 2   n_xy            982 non-null    float64
 3   n_z             982 non-null    float64
 4   n_times         982 non-null    float64
 5   pre             982 non-null    float64
 6   post_early      982 non-null    float64
 7   post_late       982 non-null    float64
 8   slice_thick     982 non-null    float64
 9   xy_spacing      982 non-null    float64
 10  mask_start      982 non-null    float64
 11  mask_end        982 non-null    float64
 12  sraw            982 non-null    float64
 13  eraw            982 non-null    float64
 14  scol            982 non-null    float64
 15  ecol            982 non-null    float64
 16  tum_vol         982 non-null    float64
 17  age             982 non-null    flo

In [83]:
df.test.value_counts(dropna=False)

0.0    784
1.0     99
2.0     99
Name: test, dtype: int64

In [84]:
df.pCR.value_counts(dropna=False)

0.0    666
1.0    316
Name: pCR, dtype: int64

In [85]:
pd.crosstab(df.test,df.pCR, dropna=False)

pCR,0.0,1.0
test,,
0.0,532,252
1.0,67,32
2.0,67,32


In [86]:
pd.crosstab(df.test,df.HR, dropna=False)

HR,0.0,1.0
test,,
0.0,351,433
1.0,45,54
2.0,50,49


In [87]:
pd.crosstab(df.test,df.HER2, dropna=False)

HER2,0.0,1.0
test,,
0.0,586,198
1.0,77,22
2.0,77,22


In [88]:
os.listdir(mask_path)[:4]

['ACRIN-6698-102212_ACRIN-6698-102212_spy2_vis1_mask.nii.gz',
 'ACRIN-6698-103939_ACRIN-6698-103939_spy2_vis1_mask.nii.gz',
 'ACRIN-6698-104268_ACRIN-6698-104268_spy2_vis1_mask.nii.gz',
 'ACRIN-6698-107700_ACRIN-6698-107700_spy2_vis1_mask.nii.gz']

In [89]:
os.listdir(dce_path)[:4]

['BreastDCEDL_ISPY2_nopcr_dce_part1',
 'BreastDCEDL_ISPY2_nopcr_dce_part2',
 'BreastDCEDL_ISPY2_nopcr_dce_part3',
 'BreastDCEDL_ISPY2_nopcr_dce_part4']

In [90]:
os.listdir(os.path.join(dce_path,os.listdir(dce_path)[5]))[:4]

['ISPY2-395379', 'ISPY2-402265', 'ISPY2-402384', 'ISPY2-405717']

## I-SPY-2 example

In [91]:
df[df.pid.isin(['ACRIN-6698-102212','ISPY2-550421'])]

,pid,pCR,n_xy,n_z,n_times,pre,post_early,post_late,slice_thick,xy_spacing,...,race_white,race_black,HR,HER2,HR_HER2_STATUS,TripleNeg,HER2pos,HRposHER2neg,dataset,test
0,ACRIN-6698-102212,0.0,256.0,80.0,8.0,0.0,2.0,6.0,2.0,0.605500,...,1.0,0.0,0.0,0.0,TripleNeg,1.0,0.0,0.0,spy2,1.0
642,ISPY2-550421,0.0,256.0,160.0,7.0,0.0,2.0,6.0,1.0,0.688073,...,0.0,1.0,0.0,0.0,TripleNeg,1.0,0.0,0.0,spy2,0.0


In [92]:
deb=0

In [ ]:

    pid='ACRIN-6698-102212'

    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])


    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    print('dce',nz,nxy,d[0].shape, 'mask: ', smask,emask,  m.shape)

    show_n_images([d[0][k] for k in range(0,nz,5)])
    show_n_images([d[1][k] for k in range(0,nz,5)])
    show_n_images([d[2][k] for k in range(0,nz,5)])
    show_n_images([m[k] for k in range(0,nz,5)])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in range(0,nz,5)])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in range(0,nz,5)])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-3,pmask,pmask+3]])
    show_n_images([ m[k] for k in [pmask-3,pmask,pmask+3]])




dce 80 256 (80, 256, 256) mask:  15 52 (80, 256, 256)


In [ ]:
    #ISPY2-905264: smask=40, emask=52, pmask=46
    pid='ISPY2-905264'
    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])

In [ ]:
    #ISPY2-905264: smask=40, emask=52, pmask=46
    pid='ISPY2-299364'
    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])

In [ ]:
    pid='ISPY2-572731'
    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([ m[k] for k in [pmask-1,pmask,pmask+1]])



In [ ]:
    pid='ISPY2-550421'

    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([ m[k] for k in [pmask-1,pmask,pmask+1]])





In [ ]:
    pid='ACRIN-6698-283070'

    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([ m[k] for k in [pmask-1,pmask,pmask+1]])





In [ ]:
    pid='ACRIN-6698-511851'

    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([ m[k] for k in [pmask-1,pmask,pmask+1]])





In [ ]:
    pid='ACRIN-6698-220471'

    row = df[df.pid==pid].iloc[0]
    smask=int(row['mask_start'])
    emask = int(row['mask_end'])
    pmask = (smask+emask)//2
    nz=int(row['n_z'])
    nxy=int(row['n_xy'])
    print('mask: ', smask,emask)

    d=get_sorted_nifti_acquisitions(pid)
    m = get_nifti_mask(pid)

    im=to_rgb(d[0],d[1], d[2])
    show_n_images([to_rgb(d[0][k],d[1][k], d[2][k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([to_rgb(d[0][k],d[1][k], m[k]) for k in [pmask-1,pmask,pmask+1]])
    show_n_images([ m[k] for k in [pmask-1,pmask,pmask+1]])



